In [1]:
import meshio

msh = meshio.read('simple.msh')

points = msh.points
cells = msh.cells

cell = cells[1].data[222]

print(f"Pointids: {cell}, This is a {cells[1].type}")

point_coords = points[cell]

print(f"Point coordinates: {point_coords}")


Pointids: [110 234 128], This is a triangle
Point coordinates: [[0.02586236 0.35555427 0.        ]
 [0.09310181 0.31363832 0.        ]
 [0.15125285 0.39462475 0.        ]]


In [ ]:
from abc import ABC, abstractmethod
import meshio

class Point:
    """
    A class representing a point in a two-dimensional space.

    Attributes:
    - x (float): The x-coordinate of the point.
    - y (float): The y-coordinate of the point.
    """
    def __init__(self, x, y):
        """
        Initializes a Point object with the given x and y coordinates.

        Parameters:
        - x (float): The x-coordinate of the point.
        - y (float): The y-coordinate of the point.
        """
        self.x = x
        self.y = y


class Cell(ABC):
    """
    An abstract base class representing a cell in a mesh.

    Attributes:
    - index (int): The unique identifier of the cell.
    - _points (list): List of Point objects representing the vertices of the cell.
    - _neighbors (list): List of indices of neighboring cells.
    """
    def __init__(self, index, points):
        """
        Initializes a Cell object.

        Parameters:
        - index (int): The unique identifier of the cell.
        - points (list): List of Point objects representing the vertices of the cell.
    
        Attributes:
        - index (int): The unique identifier of the cell.
        - _points (list): List of Point objects representing the vertices of the cell.
        - _neighbors (list): List of indices of neighboring cells.

        """
        self.index = index
        self._points = points
        self._neighbors = []

    def store_neighbors(self, cells):
        pass

    @abstractmethod
    def cell_type(self):
        pass

class Triangle(Cell):
    """
    A class representing a triangular cell in a mesh, inheriting from the Cell class.

    Attributes:
    - index (int): The unique identifier of the triangle.
    - _points (list): List of Point objects representing the vertices of the triangle.

    Methods:
    - cell_type(): Returns the type of the cell as a string. 
    - store_neighbors(cells): Stores the indices of neighboring triangles based on the given list of cells. 
    """
    def __init__(self, index, points):
        """
        Initializes a Triangle object.

        Parameters:
        - index (int): The unique identifier of the triangle.
         - points (list): List of three Point objects representing the vertices of the triangle.
    
        """
        super().__init__(index, points)

    def cell_type(self):
        """
        Returns the type of cell as a string.
        """
        return "triangle"
    
    def store_neighbors(self, cells):
        """
        Stores the indices of neighboring triangles based in the given list of cells.

        Parameters: 
        - cells(list): List of Cell objects in the mesh. 
        """
        for cell in cells:
            if isinstance(cell, Triangle) and cell.index != self.index:
                common_points = set(self._points) & set(cell._points)
                if len(common_points) == 2:
                    self._neighbors.append(cell.index)

class Line(Cell):
    """
    A class representing a line cell in a mesh, inheriting from the Cell class.

    Attributes:
    - index (int): The unique identifier of the line.
    - _points (list): List of Point objects representing the endpoints of the line.

    Methods:
    - cell_type(): Returns the type of the cell as a string.
    - store_neighbors(cells): Stores the indices of neighboring triangles based on the given list of cells.
    """
    def __init__(self, index, points):
        super().__init__(index, points)
        """
        Initializes a Line object.

        Parameters:
        - index (int): The unique identifier of the line.
        - points (list): List of two Point objects representing the endpoints of the line.
        """
    def cell_type(self):
        """
        Returns the type of the cell as a string.
        """
        return "line"
    
    def store_neighbors(self, cells):
        """
        Stores the indices of neighboring triangles based on the given list of cells.

        Parameters:
        - cells (list): List of Cell objects in the mesh.
        """
        for cell in cells:
            if isinstance(cell, Triangle) and cell.index != self.index:
                common_points = set(self._points) & set(cell._points)
                if len(common_points) == 2:
                    self._neighbors.append(cell.index)

    

        
class Mesh:
    """
    Represents a mesh consisting of points and cells.

    Attributes:
        _points (list): List of Point objects representing the mesh points.
        _cells (list): List of Cell objects representing the mesh cells.

    Methods:
        __init__(self, mesh_file):
            Initializes a Mesh object by reading data from the specified mesh file.

        _read_mesh_file(self, mesh_file):
            Reads mesh data from the given file and extracts points and cells.

        create_cell(self, cell_type, index, points):
            Creates a Cell object based on the given cell type, index, and points.

    """   
    def __init__(self, mesh_file):
        """
        Initializes a Mesh object by reading data from the specified mesh file.

        Parameters:
            mesh_file (str): The path to the mesh file.
        """
        self._points = []
        self._cells = []
        self._read_mesh_file(mesh_file)
    
    def _read_mesh_file(self, mesh_file): 
        """
        Reads mesh data from the given file and extracts points and cells.

        Parameters:
            mesh_file (str): The path to the mesh file.
        """
        mesh = meshio.read(mesh_file)

        # Extract points
        for point in mesh.points:
            self._points.append(Point(point[0], point[1]))

        # Extract cells
        for i, cell_block in enumerate(mesh.cells):
            cell_type = cell_block.type
            cell_data = cell_block.data
            cells = []

            for cell_points_indices in cell_data:
                cell_points = [self._points[index] for index in cell_points_indices]
                cell = self.create_cell(cell_type, cell_points_indices[i], cell_points)
                cells.append(cell)

            self._cells.extend(cells)

    def create_cell(self, cell_type, index, points):
        """
        Creates a Cell object based on the given cell type, index, and points.

        Parameters:
            cell_type (str): The type of the cell (e.g., "triangle", "line").
            index (int): The index of the cell.
            points (list): List of Point objects representing the cell's vertices.

        Returns:
            Cell: An instance of the Cell class corresponding to the given cell type.

        Raises:
            ValueError: If the provided cell type is unknown.
        """
        if cell_type == "triangle":
            return Triangle(index, points)
        elif cell_type == "line":
            return Line(index, points)
        else:
            # Handle unknown cell types if needed
            raise ValueError(f"Unknown cell type: {cell_type}")
        
            
    def find_neighbors(self):
        """
        Finds neighbors for each cell in the mesh and stores them using the `store_neighbors` method
        in each Cell object.

        This method iterates through each cell in the mesh and calls the `store_neighbors` method
        on each cell, providing the entire list of cells as potential neighbors. The `store_neighbors`
        method of each Cell object is responsible for identifying and storing its neighbors based on
        the given list of cells.

        Note:
        The effectiveness of this method relies on the correct implementation of the `store_neighbors`
        method in the Cell class.

        """
        for cell in self._cells:
            cell.store_neighbors(self._cells)
        

mesh = Mesh('simple.msh')
mesh.find_neighbors()

cell_ids_to_check = [4, 189, 222]

for cell_id in cell_ids_to_check:
    cell = mesh._cells[cell_id]
    neighbors_ids = cell._neighbors
    print(f"Neighbors of cell {cell_id}: {neighbors_ids}")
